# Dragonfly g/r cirrus modeling

This notebook builds a shared luminance image from source-subtracted Dragonfly g- and r-band images, applies morphology filtering, and predicts a cirrus model in each band.

In [1]:
from pathlib import Path

from dfcirrus.modeling import ModelingConfig, MultiBandModeler
from dfcirrus.modeling.diagnostics import (
    plot_interband_fits,
    plot_model_images,
    plot_morphology,
    plot_planck_fits,
)

## Input files and calibration

Set the image paths and update the calibration values if they differ from the defaults. Images are expected to contain calibrated counts with celestial WCS headers.

In [ ]:
G_IMAGE = Path("data/field-g.fits").resolve()
R_IMAGE = Path("data/field-r.fits").resolve()
PLANCK_MAP = Path("/Users/qliu/data/HFI_CompMap_ThermalDustModel_2048_R1.20.fits")
OUTPUT_DIR = Path("outputs/dragonfly_gr").resolve()

PIXEL_SCALE = 2.5  # arcsec/pixel
PSF_FWHM = 6.0     # arcsec
ZEROPOINT_G = 27.5
ZEROPOINT_R = 27.5

## Configuration

Use `rht` for morphology-based cleaning only, or `rht_starlet` to add scale-based cleanup afterward. RHT targets LSBGs, stars, and distant ICL/IGL through their morphology. Starlet suppresses selected scales associated with noise, interpolation texture, and small residual leftovers.

In [ ]:
config = ModelingConfig.from_dict({
    "run": {
        "name": "dragonfly_gr",
        "output_dir": str(OUTPUT_DIR),
        "random_seed": 12345,
    },
    "bands": {
        "g": {
            "image": str(G_IMAGE),
            "wavelength": 4770,
            "pixel_scale": PIXEL_SCALE,
            "psf_fwhm": PSF_FWHM,
            "zeropoint": ZEROPOINT_G,
        },
        "r": {
            "image": str(R_IMAGE),
            "wavelength": 6231,
            "pixel_scale": PIXEL_SCALE,
            "psf_fwhm": PSF_FWHM,
            "zeropoint": ZEROPOINT_R,
        },
    },
    "planck_radiance": {"path": str(PLANCK_MAP)},
    "alignment": {"reference_band": "r", "psf_match": True},
    "morphology": {
        "backend": "rht_starlet",
        "working_pixel_scale": 2.5,  # arcsec/pixel
        "infill": {
            "enabled": True,
            "backend": "maskfill",
            "maskfill_window_size": 9,  # pixels; must be odd
            "patch_size": 51,  # pixels; CloudCovFix only
            "training_window": 129,  # pixels; CloudCovFix only
            "conditioning_radius": 25,  # pixels; CloudCovFix only
            "memory_budget_mb": 256,  # MiB; CloudCovFix only
        },
        "rht": {
            "radius": 2,  # arcmin
            "response": "peak",
            "angle_bins": "auto",
            "median_filter_size": 3,
        },
        "starlet": {
            "scales": 5,
            "keep_scales": [2, 3, 4, 5],  # omit the first detail scale
            "threshold_sigma": 0,
            "include_coarse": True,
        },
    },
    "color": {
        "reference_band": "r",
        "bands": ["g", "r"],
        "relation": "linear",
        "regression": "bisector",
        "combine": "median",
        "bootstrap_samples": 100,
        "fit_sigma_range": [-15, 20],
    },
})
config.validate_files()

## Run the model

In [ ]:
modeler = MultiBandModeler(config)
result = modeler.run()

## Cirrus color

The reported uncertainty is estimated by bootstrapping the Planck correlations.

In [ ]:
color = result.colors["g-r"]
print(f"g-r = {color.value:.3f} +/- {color.error:.3f} mag")

## Goodness of the Planck and inter-band fits

In [ ]:
plot_planck_fits(result);

In [ ]:
plot_interband_fits(result);

## Morphology diagnostics

The panels trace the RHT morphology cleaning and, for `rht_starlet`, the subsequent scale decomposition and final filtered output.

In [ ]:
plot_morphology(result);

## Main model images

In [ ]:
plot_model_images(result);

luminance_image = result.luminance
g_model = result.models["g"]
r_model = result.models["r"]

## Save outputs

In [ ]:
result.write(OUTPUT_DIR, overwrite=True)
print(f"Saved outputs to {OUTPUT_DIR.resolve()}")